# Step 2: General Preparation (Spark Port)

In [36]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

from pyspark.sql import functions as F
from spark_utils import get_spark_session, save_parquet
from general_prep import (
    add_int_by_extracting_number,
    add_months_since_column,
    fit_dummy_spec,
    apply_dummy_columns,
    get_missing_summary,
    fill_null_with_column,
    fit_mean_imputer,
    apply_mean_imputer,
    fill_null_with_zero,
    check_null_count,
    add_good_bad_target,
    fit_median_imputer,
    apply_median_imputer
)

In [37]:
SAMPLE_MODE = False # True for quick development; False for full dataset

## 2A.1 Load Raw Base Data

In [38]:
spark = get_spark_session()
df = spark.read.parquet('/home/jovyan/work/data/parquet/accepted_raw.parquet')

if SAMPLE_MODE:
    df = df.sample(fraction=0.1, seed=42)

print(f'Rows: {df.count()}, Columns: {len(df.columns)}')

Spark driver memory: 7g
Rows: 2260701, Columns: 151


In [39]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double 

## 2A.2 Train/Test Split

In [40]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f'Train rows: {train_df.count()}')
print(f'Test rows: {test_df.count()}')

Train rows: 1807770
Test rows: 452931


## 2B.1 Continuous Variable Preprocessing

In [41]:
def transform_continuous_variables(df):
    df = add_int_by_extracting_number(
        df,
        source_col="emp_length",
        target_col="emp_length_int",
        special_map={"< 1 year": 0}
    )

    df = add_int_by_extracting_number(
        df,
        source_col="term",
        target_col="term_int"
    )

    df = add_months_since_column(
        df,
        source_col='earliest_cr_line',
        output_col='mths_since_earliest_cr_line',
        reference_date='2017-12-01',
        fmt='MMM-yyyy',
    )

    df = add_months_since_column(
        df,
        source_col='issue_d',
        output_col='mths_since_issue_d',
        reference_date='2017-12-01',
        fmt='MMM-yyyy',
    )

    return df

train_df = transform_continuous_variables(train_df)
test_df  = transform_continuous_variables(test_df)

In [42]:
cols = ['emp_length', 'emp_length_int', 'term', 'term_int', 'earliest_cr_line', 'mths_since_earliest_cr_line', 'issue_d', 'mths_since_issue_d']

train_df.select(
    *cols
).show(5, truncate=False)

train_df.select(*cols).printSchema()

test_df.select(
    *cols
).show(5, truncate=False)

test_df.select(*cols).printSchema()

+----------+--------------+----------+--------+----------------+---------------------------+--------+------------------+
|emp_length|emp_length_int|term      |term_int|earliest_cr_line|mths_since_earliest_cr_line|issue_d |mths_since_issue_d|
+----------+--------------+----------+--------+----------------+---------------------------+--------+------------------+
|9 years   |9             | 36 months|36      |Jan-2004        |167                        |Aug-2017|4                 |
|NULL      |NULL          | 36 months|36      |Aug-2006        |136                        |Mar-2018|126               |
|5 years   |5             | 36 months|36      |Jun-1995        |270                        |Feb-2015|34                |
|10+ years |10            | 36 months|36      |Oct-1983        |410                        |Jan-2015|35                |
|10+ years |10            | 36 months|36      |Aug-2006        |136                        |Jan-2015|35                |
+----------+--------------+-----

In [43]:
train_df.select(
    F.min("mths_since_earliest_cr_line").alias("min"),
    F.max("mths_since_earliest_cr_line").alias("max"),
    F.avg("mths_since_earliest_cr_line").alias("mean"),
    F.expr("percentile_approx(mths_since_earliest_cr_line, 0.5)").alias("median")
).show()

+---+----+------------------+------+
|min| max|              mean|median|
+---+----+------------------+------+
| 25|1017|215.67396903509606|   200|
+---+----+------------------+------+



In [44]:
train_df.select(
    F.min("mths_since_issue_d").alias("min"),
    F.max("mths_since_issue_d").alias("max"),
    F.avg("mths_since_issue_d").alias("mean"),
    F.expr("percentile_approx(mths_since_issue_d, 0.5)").alias("median")
).show()

+---+---+-----------------+------+
|min|max|             mean|median|
+---+---+-----------------+------+
|  0|126|48.05939450463921|    30|
+---+---+-----------------+------+



## 2B.2 Discrete Variable Dummy Columns

In [45]:
# discrete_cols = [
#     'grade',
#     'sub_grade',
#     'home_ownership',
#     'verification_status',
#     'purpose',
#     'addr_state',
#     'initial_list_status',
# ]

In [46]:
# dummy_spec = fit_dummy_spec(train_df, discrete_cols, max_categories=80)

# train_df = apply_dummy_columns(train_df, dummy_spec)
# test_df  = apply_dummy_columns(test_df, dummy_spec)

In [47]:
# train_dummy_cols = sorted([c for c in train_df.columns if ':' in c])
# test_dummy_cols = sorted([c for c in test_df.columns if ':' in c])

# print(len(train_dummy_cols), len(test_dummy_cols))
# print(train_dummy_cols == test_dummy_cols)

## 2B.3 Missing Value Treatment

In [48]:
train_missing = get_missing_summary(train_df)
test_missing = get_missing_summary(test_df)

train_missing.show(200, truncate=False)
test_missing.show(200, truncate=False)

+------------------------------------------+-------------+---------------------+
|column                                    |missing_count|missing_ratio        |
+------------------------------------------+-------------+---------------------+
|member_id                                 |1807770      |1.0                  |
|orig_projected_additional_accrued_interest|1800833      |0.9961626755616035   |
|hardship_payoff_balance_amount            |1799044      |0.9951730585196126   |
|payment_plan_start_date                   |1799044      |0.9951730585196126   |
|hardship_status                           |1799044      |0.9951730585196126   |
|hardship_length                           |1799044      |0.9951730585196126   |
|hardship_type                             |1799044      |0.9951730585196126   |
|hardship_dpd                              |1799044      |0.9951730585196126   |
|deferral_term                             |1799044      |0.9951730585196126   |
|hardship_loan_status       

In [49]:
for c in ["annual_inc", "funded_amnt"]:
    print(c)
    train_df.select(
        F.count(c).alias("count"),
        F.mean(c).alias("mean"),
        F.expr(f"percentile_approx({c}, 0.5)").alias("median"),
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.expr(f"percentile_approx({c}, array(0.01, 0.25, 0.5, 0.75, 0.99))").alias("quantiles")
    ).show(truncate=False)

annual_inc
+-------+----------------+-------+---+-----+----------------------------------------------+
|count  |mean            |median |min|max  |quantiles                                     |
+-------+----------------+-------+---+-----+----------------------------------------------+
|1807742|77997.6486033132|65000.0|0.0|1.1E8|[16788.0, 46000.0, 65000.0, 93000.0, 270000.0]|
+-------+----------------+-------+---+-----+----------------------------------------------+

funded_amnt
+-------+------------------+-------+-----+-------+-------------------------------------------+
|count  |mean              |median |min  |max    |quantiles                                  |
+-------+------------------+-------+-----+-------+-------------------------------------------+
|1807743|15036.507180500768|12800.0|500.0|40000.0|[1550.0, 8000.0, 12800.0, 20000.0, 40000.0]|
+-------+------------------+-------+-----+-------+-------------------------------------------+



In [50]:
annual_inc_mean = fit_mean_imputer(train_df, "annual_inc")

train_df = apply_mean_imputer(train_df, "annual_inc", annual_inc_mean)
test_df  = apply_mean_imputer(test_df,  "annual_inc", annual_inc_mean)

funded_amnt_median = fit_median_imputer(train_df, "funded_amnt")

train_df = apply_median_imputer(train_df, "funded_amnt", funded_amnt_median)
test_df  = apply_median_imputer(test_df,  "funded_amnt", funded_amnt_median)

In [51]:
train_df = fill_null_with_column(train_df, "total_rev_hi_lim", "funded_amnt")
test_df  = fill_null_with_column(test_df,  "total_rev_hi_lim", "funded_amnt")

In [52]:
zero_fill_cols = [
    "mths_since_earliest_cr_line",
    "acc_now_delinq",
    "total_acc",
    "pub_rec",
    "open_acc",
    "inq_last_6mths",
    "delinq_2yrs",
    "emp_length_int",
]

train_df = fill_null_with_zero(train_df, zero_fill_cols)
test_df  = fill_null_with_zero(test_df,  zero_fill_cols)

In [53]:
check_cols = [
    "total_rev_hi_lim",
    "funded_amnt",
    "annual_inc",
    "mths_since_earliest_cr_line",
    "acc_now_delinq",
    "total_acc",
    "pub_rec",
    "open_acc",
    "inq_last_6mths",
    "delinq_2yrs",
    "emp_length_int",
]

check_null_count(train_df, check_cols, "train_df after imputation")
check_null_count(test_df, check_cols, "test_df after imputation")

=== null check: train_df after imputation ===
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+
|total_rev_hi_lim|funded_amnt|annual_inc|mths_since_earliest_cr_line|acc_now_delinq|total_acc|pub_rec|open_acc|inq_last_6mths|delinq_2yrs|emp_length_int|
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+
|0               |0          |0         |0                          |0             |0        |0      |0       |0             |0          |0             |
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+

=== null check: test_df after imputation ===
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+

## 2B.4 Target Encoding (Good/Bad)

In [54]:
train_df = add_good_bad_target(train_df)
test_df  = add_good_bad_target(test_df)

In [55]:
train_df.groupBy("loan_status", "good_bad") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

test_df.groupBy("loan_status", "good_bad") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

+---------------------------------------------------+--------+------+
|loan_status                                        |good_bad|count |
+---------------------------------------------------+--------+------+
|Fully Paid                                         |1       |860720|
|Current                                            |1       |703154|
|Charged Off                                        |0       |214313|
|Late (31-120 days)                                 |0       |17129 |
|In Grace Period                                    |1       |6713  |
|Late (16-30 days)                                  |1       |3476  |
|Does not meet the credit policy. Status:Fully Paid |1       |1588  |
|Does not meet the credit policy. Status:Charged Off|0       |619   |
|Default                                            |0       |31    |
|NULL                                               |1       |27    |
+---------------------------------------------------+--------+------+

+------------------

In [56]:
train_df.groupBy("good_bad").count().orderBy("good_bad").show()
test_df.groupBy("good_bad").count().orderBy("good_bad").show()

+--------+-------+
|good_bad|  count|
+--------+-------+
|       0| 232092|
|       1|1575678|
+--------+-------+

+--------+------+
|good_bad| count|
+--------+------+
|       0| 58735|
|       1|394196|
+--------+------+



## 2B.7 Save Prepared Datasets

In [57]:
save_parquet(train_df, '/home/jovyan/work/data/parquet/accepted_general_prep_train.parquet')
save_parquet(test_df, '/home/jovyan/work/data/parquet/accepted_general_prep_test.parquet')

print('Prep parquet files saved.')

Saved to /home/jovyan/work/data/parquet/accepted_general_prep_train.parquet
Saved to /home/jovyan/work/data/parquet/accepted_general_prep_test.parquet
Prep parquet files saved.
